# Polars Exercises 06 — Joins & Merging

This is the chapter that matters most for our daily work in Foundry. Real pipelines are never one table — they are an `orders` fact table plus `customers`, `products`, and `regions` dimension tables that we **join** together to build one rich, denormalized analytics table. No more VLOOKUPs.

We will use every join type — `inner`, `left`, `full`, `semi`, `anti`, `cross` — joins on differently-named keys (`left_on`/`right_on`), joins on multiple keys, and the all-important habit of **checking row counts** so a join never silently explodes our data.


**Data files:** `data/orders.csv`, `data/customers.csv`, `data/products.csv`, `data/regions.csv`

---

### 1. Import Polars and read all four tables: `orders`, `customers`, `products`, `regions`.

In [1]:
import polars as pl

orders = pl.read_csv("data/orders.csv")
customers = pl.read_csv("data/customers.csv")
products = pl.read_csv("data/products.csv")
regions = pl.read_csv("data/regions.csv")
orders.shape, customers.shape, products.shape, regions.shape

((1000, 12), (200, 5), (12, 5), (5, 4))

### 2. Look at the **keys**: print the columns of each table. Which column links `orders` to `customers`? to `products`?

In [2]:
for name, df in [("orders", orders), ("customers", customers),
                 ("products", products), ("regions", regions)]:
    print(name, "->", df.columns)

orders -> ['order_id', 'customer_id', 'order_date', 'region', 'product_sku', 'category', 'product_name', 'quantity', 'unit_price', 'channel', 'status', 'discount_code']
customers -> ['customer_id', 'customer_name', 'region_id', 'segment', 'signup_date']
products -> ['product_sku', 'product_name', 'category', 'unit_cost', 'list_price']
regions -> ['region_id', 'region_name', 'currency', 'tax_rate']


### 3. **Left join** `orders` with `customers` on `customer_id` to attach each order's `segment` and `signup_date`.

In [3]:
orders.join(
    customers.select("customer_id", "segment", "signup_date"),
    on="customer_id", how="left",
).head()

order_id,customer_id,order_date,region,product_sku,category,product_name,quantity,unit_price,channel,status,discount_code,segment,signup_date
str,str,str,str,str,str,str,i64,f64,str,str,str,str,str
"""ORD-000001""","""CUST-0171""","""2023-10-25""","""Asia Pacific""","""PB200""","""pens""","""Gel Pen 12-pack""",15,2.7,"""online""","""completed""","""SPRING10""","""SMB""","""2021-07-09"""
"""ORD-000002""","""CUST-0049""","""2023-12-02""","""North America""","""SH900""","""shirts""","""Cotton T-Shirt""",5,17.88,"""online""","""completed""","""BULK""","""SMB""","""2019-04-20"""
"""ORD-000003""","""CUST-0060""","""2023-02-27""","""Europe""","""BK300""","""books""","""Paperback Book""",16,7.89,"""online""","""completed""","""BULK""","""Mid-Market""","""2021-03-17"""
"""ORD-000004""","""CUST-0065""","""2023-12-26""","""Latin America""","""SH900""","""shirts""","""Cotton T-Shirt""",37,17.06,"""online""","""completed""","""BULK""","""Mid-Market""","""2018-12-15"""
"""ORD-000005""","""CUST-0036""","""2023-05-24""","""Asia Pacific""","""PT010""","""posters""","""Motivational Poster""",11,7.26,"""online""","""completed""",null,"""SMB""","""2019-05-21"""


### 4. A left join must **never add rows**. Confirm that the row count after the join equals the original `orders` row count.

In [4]:
joined = orders.join(customers, on="customer_id", how="left")
orders.height, joined.height, orders.height == joined.height

(1000, 1000, True)

### 5. **Inner join** `orders` with `products` on `product_sku` to attach `unit_cost` and `list_price`.

In [5]:
orders.join(
    products.select("product_sku", "unit_cost", "list_price"),
    on="product_sku", how="inner",
).head()

order_id,customer_id,order_date,region,product_sku,category,product_name,quantity,unit_price,channel,status,discount_code,unit_cost,list_price
str,str,str,str,str,str,str,i64,f64,str,str,str,f64,f64
"""ORD-000001""","""CUST-0171""","""2023-10-25""","""Asia Pacific""","""PB200""","""pens""","""Gel Pen 12-pack""",15,2.7,"""online""","""completed""","""SPRING10""",1.1,3.0
"""ORD-000002""","""CUST-0049""","""2023-12-02""","""North America""","""SH900""","""shirts""","""Cotton T-Shirt""",5,17.88,"""online""","""completed""","""BULK""",5.5,17.0
"""ORD-000003""","""CUST-0060""","""2023-02-27""","""Europe""","""BK300""","""books""","""Paperback Book""",16,7.89,"""online""","""completed""","""BULK""",3.0,9.0
"""ORD-000004""","""CUST-0065""","""2023-12-26""","""Latin America""","""SH900""","""shirts""","""Cotton T-Shirt""",37,17.06,"""online""","""completed""","""BULK""",5.5,17.0
"""ORD-000005""","""CUST-0036""","""2023-05-24""","""Asia Pacific""","""PT010""","""posters""","""Motivational Poster""",11,7.26,"""online""","""completed""",null,2.3,7.0


### 6. After joining `products`, compute each order's **margin** = `(unit_price - unit_cost) * quantity`. Show the 5 columns `order_id, quantity, unit_price, unit_cost, margin`.

In [6]:
orders.join(products.select("product_sku", "unit_cost"), on="product_sku")\
    .with_columns(
        margin=(pl.col("unit_price") - pl.col("unit_cost")) * pl.col("quantity")
    ).select("order_id", "quantity", "unit_price", "unit_cost", "margin").head()

order_id,quantity,unit_price,unit_cost,margin
str,i64,f64,f64,f64
"""ORD-000001""",15,2.7,1.1,24.0
"""ORD-000002""",5,17.88,5.5,61.9
"""ORD-000003""",16,7.89,3.0,78.24
"""ORD-000004""",37,17.06,5.5,427.72
"""ORD-000005""",11,7.26,2.3,54.56


### 7. Join on **differently-named keys**: `orders.region` holds a region *name*, while `regions.region_name` holds the same names. Left-join them (`left_on` / `right_on`) to attach `currency` and `tax_rate`.

In [7]:
orders.join(
    regions.select("region_name", "currency", "tax_rate"),
    left_on="region", right_on="region_name", how="left",
).select("order_id", "region", "currency", "tax_rate").head()

order_id,region,currency,tax_rate
str,str,str,f64
"""ORD-000001""","""Asia Pacific""","""USD""",0.1
"""ORD-000002""","""North America""","""USD""",0.0
"""ORD-000003""","""Europe""","""EUR""",0.2
"""ORD-000004""","""Latin America""","""USD""",0.15
"""ORD-000005""","""Asia Pacific""","""USD""",0.1


### 8. **Semi join:** which customers placed **at least one** order? Return the matching customer rows only (no order columns added).

In [8]:
customers.join(orders, on="customer_id", how="semi").head()

customer_id,customer_name,region_id,segment,signup_date
str,str,str,str,str
"""CUST-0001""","""Zooxo Partners""","""APAC""","""SMB""","""2021-07-07"""
"""CUST-0002""","""Yodel Co""","""MEA""","""Enterprise""","""2019-11-29"""
"""CUST-0003""","""Mybuzz Systems""","""EU""","""Mid-Market""","""2021-12-08"""
"""CUST-0004""","""Zooxo Holdings""","""NA""","""SMB""","""2021-01-16"""
"""CUST-0005""","""Skipfire Inc""","""NA""","""Enterprise""","""2020-08-08"""


### 9. **Anti join:** which customers placed **no** orders at all?

In [9]:
customers.join(orders, on="customer_id", how="anti")

customer_id,customer_name,region_id,segment,signup_date
str,str,str,str,str
"""CUST-0186""","""Photobean Holdings""","""MEA""","""Mid-Market""","""2018-12-10"""


### 10. **How many** customers have no orders? (count the anti-join rows)

In [10]:
customers.join(orders, on="customer_id", how="anti").height

1

### 11. Build a **multi-key join**. First make two summaries per `region` + `channel`: total revenue, and order count. Then **join them on both keys** (`on=["region", "channel"]`).

In [11]:
rev = (orders.with_columns(revenue=pl.col("quantity") * pl.col("unit_price"))
       .group_by("region", "channel").agg(pl.col("revenue").sum()))
cnt = orders.group_by("region", "channel").agg(pl.len().alias("n_orders"))
rev.join(cnt, on=["region", "channel"]).sort("revenue", descending=True)

region,channel,revenue,n_orders
str,str,f64,u32
"""North America""","""online""",37005.36,224
"""North America""","""store""",26755.07,138
"""Europe""","""online""",24065.12,145
"""Asia Pacific""","""online""",21491.05,144
"""Europe""","""store""",13200.43,72
…,…,…,…
"""Europe""","""partner""",3423.87,24
"""Middle East & Africa""","""online""",3353.37,19
"""Middle East & Africa""","""store""",2017.86,11


### 12. Join `products` (for `unit_cost`) and compute the **total margin per category** (join, then `group_by` + `agg`).

In [12]:
orders.join(products.select("product_sku", "unit_cost"), on="product_sku")\
    .with_columns(
        margin=(pl.col("unit_price") - pl.col("unit_cost")) * pl.col("quantity")
    ).group_by("category").agg(pl.col("margin").sum())\
    .sort("margin", descending=True)

category,margin
str,f64
"""shirts""",35768.54
"""books""",29666.32
"""posters""",18861.7
"""notebooks""",12231.91
"""mugs""",6722.92
"""pens""",3681.12
"""stickers""",2043.25


### 13. Join `customers` and report the **total revenue of Enterprise-segment orders** (join, filter `segment == "Enterprise"`, sum revenue).

In [13]:
orders.join(customers.select("customer_id", "segment"), on="customer_id")\
    .filter(pl.col("segment") == "Enterprise")\
    .select((pl.col("quantity") * pl.col("unit_price")).sum().alias("enterprise_revenue"))

enterprise_revenue
f64
26096.27


### 14. 🎯 **Mission — build the enriched analytics table.** Chain the joins: `orders` + `customers` (segment) + `products` (unit_cost) + `regions` (tax_rate). Then add `revenue` and `margin`. Show the first rows.

In [14]:
analytics = (
    orders
    .join(customers.select("customer_id", "segment"), on="customer_id", how="left")
    .join(products.select("product_sku", "unit_cost"), on="product_sku", how="left")
    .join(regions.select("region_name", "tax_rate"),
          left_on="region", right_on="region_name", how="left")
    .with_columns(
        revenue=pl.col("quantity") * pl.col("unit_price"),
        margin=(pl.col("unit_price") - pl.col("unit_cost")) * pl.col("quantity"),
    )
)
analytics.select(
    "order_id", "segment", "category", "tax_rate", "revenue", "margin"
).head()

order_id,segment,category,tax_rate,revenue,margin
str,str,str,f64,f64,f64
"""ORD-000001""","""SMB""","""pens""",0.1,40.5,24.0
"""ORD-000002""","""SMB""","""shirts""",0.0,89.4,61.9
"""ORD-000003""","""Mid-Market""","""books""",0.2,126.24,78.24
"""ORD-000004""","""Mid-Market""","""shirts""",0.15,631.22,427.72
"""ORD-000005""","""SMB""","""posters""",0.1,79.86,54.56


### 15. Using the enriched table, compute `revenue_with_tax` = `revenue * (1 + tax_rate)`, rounded to 2 decimals.

In [15]:
analytics.with_columns(
    revenue_with_tax=(pl.col("revenue") * (1 + pl.col("tax_rate"))).round(2)
).select("order_id", "revenue", "tax_rate", "revenue_with_tax").head()

order_id,revenue,tax_rate,revenue_with_tax
str,f64,f64,f64
"""ORD-000001""",40.5,0.1,44.55
"""ORD-000002""",89.4,0.0,89.4
"""ORD-000003""",126.24,0.2,151.49
"""ORD-000004""",631.22,0.15,725.9
"""ORD-000005""",79.86,0.1,87.85


### 16. From the enriched table, the **top 5 customers by total margin** (group by `customer_id`).

In [16]:
analytics.group_by("customer_id").agg(pl.col("margin").sum())\
    .sort("margin", descending=True).head(5)

customer_id,margin
str,f64
"""CUST-0123""",1528.7
"""CUST-0064""",1487.46
"""CUST-0079""",1391.68
"""CUST-0107""",1333.18
"""CUST-0075""",1307.95


### 17. **Cross join** to build a template of every possible `region_name` × `category` combination (every region paired with every category).

In [17]:
regions.select("region_name").join(
    products.select("category").unique(), how="cross"
).sort("region_name", "category")

region_name,category
str,str
"""Asia Pacific""","""books"""
"""Asia Pacific""","""mugs"""
"""Asia Pacific""","""notebooks"""
"""Asia Pacific""","""pens"""
"""Asia Pacific""","""posters"""
…,…
"""North America""","""notebooks"""
"""North America""","""pens"""
"""North America""","""posters"""


### 18. A left join can create **nulls** when there is no match. Left-join `orders` to **only the Enterprise customers**, then count how many orders got a null `segment` (i.e. their customer was not Enterprise).

In [18]:
ent = customers.filter(pl.col("segment") == "Enterprise").select("customer_id", "segment")
left = orders.join(ent, on="customer_id", how="left")
left.filter(pl.col("segment").is_null()).height

859